# Tutorial 2: Triple Modular Redundancy for Fault Tolerance

**Objective:** Compare fault tolerance strategies (no protection, full TMR, selective TMR, checkpoint rollback) and understand the compute-vs-reliability tradeoff.

## Background

**Triple Modular Redundancy (TMR)** runs three copies of a model in parallel and uses majority voting to mask faults. If one copy is corrupted, the other two outvote it.

The tradeoff: TMR costs 3x compute and power. On a power-constrained satellite, that may not be feasible. **Selective TMR** protects only the most vulnerable layers, saving compute while recovering most accuracy.

In this tutorial you will:
1. Measure baseline accuracy and fault-degraded accuracy
2. Apply full TMR and measure recovery
3. Apply selective TMR and compare compute cost vs recovery
4. Explore checkpoint rollback as a lightweight alternative
5. Build a cost-benefit table

In [ ]:
import copy

import torch
import torch.nn as nn

from space_ml_sim.environment.radiation import RadiationEnvironment
from space_ml_sim.compute.fault_injector import FaultInjector
from space_ml_sim.compute.tmr import TMRWrapper
from space_ml_sim.models.chip_profiles import TRILLIUM_V6E

## Step 1: Setup model and fault injector

In [ ]:
class SmallNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(20, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        return self.layers(x)

def model_factory():
    return SmallNet()

rad_env = RadiationEnvironment.leo_500km()
injector = FaultInjector(rad_env=rad_env, chip_profile=TRILLIUM_V6E)

FAULTS = 30
TRIALS = 20

# Fixed test data
torch.manual_seed(42)
test_input = torch.randn(64, 20)
baseline_model = model_factory().eval()
with torch.no_grad():
    baseline_preds = baseline_model(test_input).argmax(dim=1)

print(f"Baseline predictions computed for {test_input.shape[0]} samples")

## Step 2: Measure unprotected accuracy

In [ ]:
def measure_match_rate(model_or_tmr, is_tmr=False):
    """Average prediction match rate over multiple trials."""
    total = 0.0
    for _ in range(TRIALS):
        with torch.no_grad():
            if is_tmr:
                result = model_or_tmr.forward(test_input)
                preds = result["predictions"]
            else:
                preds = model_or_tmr(test_input).argmax(dim=1)
        total += (preds == baseline_preds).float().mean().item()
    return total / TRIALS

# Unprotected: inject faults into a single model
unprotected_rates = []
for _ in range(TRIALS):
    m = copy.deepcopy(baseline_model)
    injector.inject_weight_faults(m, num_faults=FAULTS)
    with torch.no_grad():
        preds = m(test_input).argmax(dim=1)
    unprotected_rates.append((preds == baseline_preds).float().mean().item())

avg_unprotected = sum(unprotected_rates) / len(unprotected_rates)
print(f"Unprotected match rate ({FAULTS} faults): {avg_unprotected:.2%}")

## Step 3: Full TMR

In [ ]:
full_tmr = TMRWrapper(model_factory, strategy="full_tmr")
full_tmr.inject_faults_to_replicas(injector, faults_per_replica=FAULTS)
full_tmr_rate = measure_match_rate(full_tmr, is_tmr=True)
print(f"Full TMR match rate: {full_tmr_rate:.2%}  (compute cost: 3x)")

## Step 4: Selective TMR

Selective TMR protects only the layers identified as most vulnerable to bit-flips.

In [ ]:
selective_tmr = TMRWrapper(model_factory, strategy="selective_tmr")
selective_tmr.inject_faults_to_replicas(injector, faults_per_replica=FAULTS)
selective_rate = measure_match_rate(selective_tmr, is_tmr=True)
print(f"Selective TMR match rate: {selective_rate:.2%}  (compute cost: ~1.5x)")

## Step 5: Checkpoint rollback

In [ ]:
ckpt_tmr = TMRWrapper(model_factory, strategy="checkpoint_rollback")
injector.inject_weight_faults(ckpt_tmr.model, num_faults=FAULTS)
ckpt_rate = measure_match_rate(ckpt_tmr, is_tmr=True)
print(f"Checkpoint rollback match rate: {ckpt_rate:.2%}  (compute cost: 1x + storage)")

## Step 6: Cost-benefit comparison

In [ ]:
import plotly.graph_objects as go

strategies = {
    'No Protection': (avg_unprotected, 1.0),
    'Checkpoint Rollback': (ckpt_rate, 1.1),
    'Selective TMR': (selective_rate, 1.5),
    'Full TMR': (full_tmr_rate, 3.0),
}

fig = go.Figure()
for name, (rate, cost) in strategies.items():
    fig.add_trace(go.Scatter(
        x=[cost], y=[rate * 100],
        mode='markers+text',
        text=[name],
        textposition='top center',
        marker=dict(size=14),
        name=name,
    ))

fig.update_layout(
    title=f'Fault Tolerance: Accuracy vs Compute Cost ({FAULTS} faults)',
    xaxis_title='Compute Cost (multiplier)',
    yaxis_title='Prediction Match Rate (%)',
    yaxis_range=[0, 105],
    xaxis_range=[0.5, 3.5],
    template='plotly_white',
    showlegend=False,
)
fig.show()

## Exercise

1. **Increase faults to 100.** At what point does Full TMR start to fail?
2. **Try a larger model** (ResNet-18 from torchvision). Does selective TMR still offer a good tradeoff?
3. **Power budget analysis:** If your satellite has 15W available and the chip TDP is 5W, can you run Full TMR? What about Selective TMR?
4. **Combine strategies:** What happens if you use selective TMR *and* checkpoint rollback together?